# TACO YOLO26 Training (Google Colab)

Pre-training notebook used to fine-tune the YOLO26 models for the thesis
*"Benchmarking ML Inference Strategies on Edge Devices"* (TU Wien).

This is the notebook used to fine-tune the YOLO26 models on Google Colab (T4 GPU runtime), lightly cleaned for release.
It downloads the [TACO](https://github.com/pedropro/TACO) dataset, collapses all
annotations to a single `litter` class, converts to YOLO format with an 80/10/10
train/val/test split (`seed=42`), and fine-tunes a YOLO26 model for 50 epochs.

Notes for the reader:
- The three benchmarked weights (`yolo26n` / `yolo26s` / `yolo26m`) were each produced by
  changing the `used_model` variable in the training and testing cells and re-running.
- The inline comment "Cloud tier (AWS)" reflects an earlier plan; the final cloud tier
  is a **Hugging Face Space (NVIDIA T4)**, not AWS. Kept here as originally written.
- The three fine-tuned weights and their tier mapping are described in [`../models/README.md`](../models/README.md).
- The exported `taco_test.zip` is the fixed 150-image test split used across all benchmark
  configurations (`data/content/taco_yolo/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install "ultralytics>=8.4.0" -q

# Download Taco Dataset

This can take some time just be patient.

In [ ]:
import os

if not os.path.exists('/content/TACO/data/annotations.json'):
    !git clone https://github.com/pedropro/TACO.git
    %cd TACO
    !pip install -r requirements.txt -q
    !python download.py

# Convert TACO to YOLO format (official Ultralytics converter)

Converts the COCO annotations with `ultralytics.data.converter.convert_coco`, maps all
60 categories to a single `litter` class (TACO-1), and creates the seeded 80/10/10
split with flattened file names.


In [ ]:
import json, random, shutil
from pathlib import Path
from ultralytics.data.converter import convert_coco

ANN_PATH = Path('/content/TACO/data/annotations.json')
IMG_DIR  = Path('/content/TACO/data')
ANN_FLAT_DIR = Path('/content/ann_flat')
CONV_DIR = Path('/content/taco_converted')
OUT_DIR  = Path('/content/taco_yolo')

# fresh dirs: convert_coco APPENDS to existing label files on re-runs
for d in (ANN_FLAT_DIR, CONV_DIR, OUT_DIR):
    if d.exists():
        shutil.rmtree(d)
ANN_FLAT_DIR.mkdir(parents=True)

coco = json.loads(ANN_PATH.read_text())

# flatten batch_x/yyyy.jpg -> batch_x_yyyy.jpg in the annotations COPY
# (TACO basenames repeat across batches and would collide in the converter)
for img in coco['images']:
    img['file_name'] = img['file_name'].replace('/', '_')
(ANN_FLAT_DIR / 'annotations.json').write_text(json.dumps(coco))

convert_coco(labels_dir=str(ANN_FLAT_DIR), save_dir=str(CONV_DIR), cls91to80=False)

LABELS_DIR = CONV_DIR / 'labels' / 'annotations'

# TACO-1 single-class formulation: force every class id to 0
# (also repairs the converter's category_id - 1 shift for TACO's ids)
for lf in LABELS_DIR.glob('*.txt'):
    lines = []
    for line in lf.read_text().splitlines():
        parts = line.split()
        if parts:
            parts[0] = '0'
            lines.append(' '.join(parts))
    lf.write_text('\n'.join(lines))

# seeded 80/10/10 split over the images present on disk (original file names)
coco_orig = json.loads(ANN_PATH.read_text())
img_id_to_info = {i['id']: i for i in coco_orig['images']}
valid_ids = [i for i, info in img_id_to_info.items() if (IMG_DIR / info['file_name']).exists()]
print('Images found on disk:', len(valid_ids))

random.seed(42)
random.shuffle(valid_ids)
n = len(valid_ids)
splits = {
    'train': valid_ids[:int(n*0.8)],
    'val':   valid_ids[int(n*0.8):int(n*0.9)],
    'test':  valid_ids[int(n*0.9):],
}
print('Split sizes:', {k: len(v) for k, v in splits.items()})

for split, ids in splits.items():
    (OUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)
    for img_id in ids:
        info = img_id_to_info[img_id]
        flat = info['file_name'].replace('/', '_')
        shutil.copy(IMG_DIR / info['file_name'], OUT_DIR / split / 'images' / flat)
        src_label = LABELS_DIR / (Path(flat).stem + '.txt')
        dst_label = OUT_DIR / split / 'labels' / (Path(flat).stem + '.txt')
        if src_label.exists():
            shutil.copy(src_label, dst_label)
        else:
            dst_label.write_text('')

print('Conversion done')


In [ ]:
yaml_content = """path: /content/taco_yolo
train: train/images
val:   val/images
test:  test/images

nc: 1
names:
  0: litter
"""

with open('/content/taco_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

## Zip and download test split

In [ ]:
# zip test split
!zip -r taco_test.zip /content/taco_yolo/test

In [ ]:
from google.colab import files
files.download('taco_test.zip')

# Train YOLO model on TACO

Choose different models according to https://github.com/ultralytics/ultralytics

In [ ]:
from ultralytics import YOLO

n = 'yolo26n'  # nano  → Device tier (Raspberry Pi)
s = 'yolo26s'  # small → Edge tier   (Lenovo Mini PC)
m = 'yolo26m'  # medium → Cloud tier  (AWS)
l = 'yolo26l'
x = 'yolo26x'

used_model = n

model = YOLO(f'{used_model}.pt')

results = model.train(
    data     = '/content/taco_yolo/data.yaml',
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    lr0      = 0.001,
    device   = 0,
    seed     = 42,
    project  = '/content/drive/MyDrive/litter_detection',
    name     = used_model,
    exist_ok = True,
)

print(f"mAP50:    {results.results_dict['metrics/mAP50(B)']:.3f}")
print(f"mAP50-95: {results.results_dict['metrics/mAP50-95(B)']:.3f}")

# Testing model

In [ ]:
from ultralytics import YOLO

n = 'yolo26n'  # nano  → Device tier (Raspberry Pi)
s = 'yolo26s'  # small → Edge tier   (Lenovo Mini PC)
m = 'yolo26m'  # medium → Cloud tier  (AWS)

used_model = n

print(f"Model used for test {used_model}")

model = YOLO(f'/content/drive/MyDrive/litter_detection/{used_model}/weights/best.pt')

metrics = model.val(
    data='/content/taco_yolo/data.yaml',
    split='test',
    verbose=True
)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")